In [0]:
# TASK 3 — Silver

from pyspark.sql.functions import col, to_date, sum
from pyspark.sql.window import Window

orders = spark.table("de_workspace26.michaelBronze.orders").dropDuplicates(["order_id"])

orders = orders.withColumn("order_date", to_date(col("order_date"))) \
               .withColumn("revenue", col("quantity") * col("unit_price"))

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

orders = orders.withColumn("cumulative_revenue", sum("revenue").over(window_spec))

customers = spark.table("de_workspace26.michaelBronze.customers").drop("_ingested_at", "_source_file")
products = spark.table("de_workspace26.michaelBronze.products").drop("_ingested_at", "_source_file")

orders = orders.join(customers, "customer_id", "left") \
               .join(products, "product_id", "left")

orders.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .saveAsTable("de_workspace26.michaelSilver.orders")

In [0]:
%sql
ALTER TABLE de_workspace26.michaelSilver.orders
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
OPTIMIZE de_workspace26.michaelSilver.orders
ZORDER BY (customer_id, order_date);

In [0]:
%sql
show tables in de_workspace26.michaelSilver

In [0]:
%sql
select * from de_workspace26.michaelsilver.orders limit 1;